# 03 Centrality, vulnerability score and disruption scenarios

1. Betweenness, closeness and degree centrality on the undirected network, plus weekday trips per stop.
2. The first principal component of the standardized metrics, rescaled to 0–100, is the vulnerability score. Its sign is oriented so that higher always means more central.
3. Stops are removed in scenarios and the largest connected component is measured again.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
(DATA / "processed").mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import networkx as nx
from src import centrality, disruption_simulator as ds, viz

G = nx.read_graphml(DATA / "processed" / "weekday_network.graphml")
trips = pd.read_csv(DATA / "processed" / "trips_per_stop.csv", index_col=0)["daily_trips"]
for n, d in G.nodes(data=True):
    d["lat"], d["lon"] = float(d["lat"]), float(d["lon"])

# Exact betweenness takes a few minutes on ~3,500 stops; set betweenness_k=500 for a quick sampled run.
metrics = centrality.compute_centrality(G, trips, betweenness_k=None)
scores, info = centrality.vulnerability_score(metrics)
info

In [ ]:
scores.head(20)[["rank", "stop_name", "vulnerability_score", "betweenness", "closeness", "degree", "daily_trips"]]

In [ ]:
scores["tier"].value_counts()

## Disruption scenarios
Reachability is the share of stops still in the largest connected component. The flood scenario needs the FEMA Zone AE polygon (`ds.stops_in_polygon`). The 0.5-mile power-outage buffer selects 89 stops from the published stop list, against 96 in the published scenario, so the original used a slightly different centre or distance method.

In [ ]:
gov = scores[scores.stop_name.str.contains("Government Square", case=False)].stop_id.tolist()
power = ds.stops_within(scores, 39.1017, -84.5105, 0.5)      # 0.5 mi around the centre of the six Government Square bays
scenarios = {
    "Government Square shutdown": gov,
    "Downtown power outage": power,
    "Top 5 critical stops fail": scores.stop_id.head(5).tolist(),
}
ds.run_scenarios(G, scenarios)

In [ ]:
cascade = ds.cascading_removal(G, scores.stop_id.tolist(), max_n=10)
viz.cascade_line(cascade)

In [ ]:
scores.to_csv(DATA / "processed" / "centrality_vulnerability.csv", index=False)
m = viz.vulnerability_map(scores)
(ROOT / "outputs").mkdir(exist_ok=True)
m.save(str(ROOT / "outputs" / "cincinnati_vulnerability_map.html"))
m